In [38]:
import pandas as pd
import os
from rapidfuzz import process, fuzz
import unicodedata
import requests
import csv

### Util functions


In [ ]:
def save_fpl_players(fpl_subfolders, season):

    for folder in fpl_subfolders:
        player_name = folder.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        player_df = pd.read_csv(str(folder+'/gw.csv'))
        player_dir = './data/joint/'+str(season)+'/fpl/'

         # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        player_df.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned 20'+ season +' fpl data')

# Understat Files
def save_under_players(understat_files, season):
    for file_ in understat_files:
        player_name = file_.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        player_df = pd.read_csv(str(file_))
        player_dir = './data/joint/'+str(season)+'/understat/'

        # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        player_df.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned understat 20'+ season +' data')

def joint_players_info(fpl_player_folder_path, understat_player_folder_path, season):
    fpl_subfolders = [ f.path for f in os.scandir(fpl_player_folder_path) if f.is_dir() ]
    under_files = [ f.path for f in os.scandir(understat_player_folder_path) if f.is_file() ]

    save_fpl_players(fpl_subfolders, season)
    save_under_players(under_files, season)

    print('20'+ season +' fpl and understat data now in `joint` folder')

In [4]:
def merge_fpl_understat_data(fpl_player_folder_path, understat_player_folder_path, season):
    joint_players_info(fpl_player_folder_path, understat_player_folder_path, season)

    joint_fpl_data_path = "./data/joint/"+ season + "/fpl/"
    joint_understat_path = "./data/joint/"+ season + "/understat/"

    understat_files = next(os.walk("./data/joint/"+ season + "/understat/"), (None, None, []))[2]  # [] if no file
    fpl_files = next(os.walk( "./data/joint/"+ season +"/fpl"), (None, None, []))[2]  # [] if no file
    player_ids = pd.read_csv('./data/id_dict_'+ season +'.csv')  #('./data/20'+ season +'/id_dict.csv')

    understat_names = [file_.split('.')[0] for file_ in understat_files]
    fpl_file_names = [file_.split('.')[0] for file_ in fpl_files]

    for name in fpl_file_names:
        fpl_player = player_ids[player_ids['FPL_Name'] == name]
        fpl_player['FPL_Name'].values

        if fpl_player['FPL_Name'].values.size > 0:
            fpl_player_name = fpl_player['FPL_Name'].values[0]
            understat_player_name = fpl_player['Understat_Name'].values[0]

            fpl_player_data = pd.read_csv(joint_fpl_data_path + fpl_player_name+ '.csv')
            understat_player_data = pd.read_csv(joint_understat_path + understat_player_name + '.csv')

            # Change 'kickoff_time' column name to 'date
            fpl_player_data = fpl_player_data.rename(columns={'kickoff_time': 'date'})
            # change the formats: From 2021-10-03T13:00:00Z to 2021-10-03
            fpl_player_data.date = fpl_player_data.date.apply(lambda x: x.split('T')[0])

            # Dates are of the form 2021-10-03T13:00:00Z
            fpl_dates_min = fpl_player_data['date'].min()
            fpl_dates_max = fpl_player_data['date'].max()


            # Filter out player info not in the range of dates we are dealing with
            understat_filtered = understat_player_data[(pd.to_datetime(understat_player_data['date']) >= pd.to_datetime(fpl_dates_min))
                                                        & (pd.to_datetime(understat_player_data['date']) <= pd.to_datetime(fpl_dates_max) )]

            # Marge fpl_player_data with understat_player_data if the dates match
            player_data_merged = fpl_player_data.merge(understat_filtered, on="date")

            # Add player team
            def set_player_team(row):
                return row['h_team'] if row['was_home'] else row['a_team']

            with_team = player_data_merged.apply(set_player_team, axis=1)
            player_data_merged['player_team'] = player_data_merged.apply(set_player_team, axis=1)


            if(player_data_merged.shape[0]):
                merged_dir = './data/joint/'+ season +'/merged/'
                if not os.path.exists(merged_dir):
                    os.makedirs(merged_dir)

                player_data_merged.to_csv(merged_dir+ fpl_player_name +'.csv', index_label=False )

In [5]:
def add_difficulty(season):
    print('====> Starting to add difficulty features to 20'+season)
    merged = './data/joint/' + season + '/merged/'

    player_names = next(os.walk((merged), (None, None, [])))[2]
    fixtures = pd.read_csv('./data/20' + season + '/fixtures.csv')

    # Loop over each player file in player_names
    for name in player_names:
        # Load player data
        player = pd.read_csv('./data/joint/' + season + '/merged/' + name)

        # Function to get the difficulty and was_home columns based on the fixture
        def get_fixture_info(row):
            # Filter the relevant fixture
            fixture = fixtures[fixtures['id'] == row['fixture']]
            if not fixture.empty:
                fixture = fixture.iloc[0]  # Get the first (and only) match

                # Get the team difficulties
                team_h_difficulty = fixture['team_h_difficulty']
                team_a_difficulty = fixture['team_a_difficulty']
                event = fixture['event']

                return pd.Series([team_h_difficulty, team_a_difficulty, event])
            else:
                # Return NaN if no matching fixture found
                return pd.Series([None, None, None])

        # Apply the function to each row of player
        player[['team_h_difficulty', 'team_a_difficulty', 'event']] = player.apply(get_fixture_info, axis=1)
        # Update the value to prices
        # player['value'] = player['value']/10

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/' + season + '/merged_extras/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + name, index=False)

In [52]:
def add_xP(season, gwk=None):
    print('================> starting to add xp for season 20'+season)

    # Ensure the data is cleaned for merged_gw.csv for the current gwk
    # Define the column index for 'minutes' (0-based index)
    # Define the expected number of columns
    EXPECTED_COLUMNS = 42  # Change this to the correct number of columns
    MINUTES_COLUMN_INDEX = pd.read_csv(f'./data/20{season}/gws/merged_gw.csv', on_bad_lines='skip').columns.tolist().index('minutes')
    FEATURES_TO_REMOVE = 7  # Number of features to delete after 'minutes'

    # Input and output file paths
    data_file = f'./data/20{season}/gws/merged_gw.csv'
    output_file = "cleaned_file.csv"

    with open(data_file, "r", newline="", encoding="utf-8") as infile, \
        open(output_file, "w", newline="", encoding="utf-8") as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        # Read and write the header row
        header = next(reader)
        writer.writerow(header)

        for row in reader:
            # print(row)
            if len(row) > EXPECTED_COLUMNS:
                row = row[:MINUTES_COLUMN_INDEX + 1] + row[MINUTES_COLUMN_INDEX + FEATURES_TO_REMOVE + 1:]
            writer.writerow(row)

    # Update the data file path
    outfile = pd.read_csv(output_file)

    # Save the cleaned file
    output_file = './data/20'+ '24-25' +'/gws/merged_gw.csv'
    outfile.to_csv(output_file, index=False)
    print(f"Cleaned file saved as {output_file}")


    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        player_data = None
        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            players = data['elements']  # Extract the list of players

            player_data = [
                {"id": player["id"],  "name": f"{player['first_name']} {player['second_name']}", "player_team": player["team"], "price": player["now_cost"] / 10, "position": player["element_type"]}
                for player in players
            ]

        else:
            print('Failed to retrieve data')

        fpl_players = pd.DataFrame(player_data)

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras', [None], [None],[]))[2]
    # players_paths
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras/'+ path)
        merged = pd.read_csv('./data/20'+ season +'/gws/merged_gw.csv')

        player = player.drop(['position'], axis=1)
        merged_player = pd.merge(player, merged[['element', 'fixture', 'xP','position']], on=['element', 'fixture'], how='left')


        if gwk:
            merged_player = merged_player.reindex(merged_player.index.tolist()  + list([merged_player.index[-1]+1]))


            player_id = int(merged_player.iloc[-2, merged_player.columns.get_loc('element')])
            fpl_data = fpl_players[fpl_players['id'] == player_id]

            merged_player.iloc[-1, merged_player.columns.get_loc('event')] = gwk
            merged_player.iloc[-1, merged_player.columns.get_loc('value')] = fpl_data['price'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('position')] = fpl_data['position'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('fpl_id')] = player['fpl_id'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('understat_id')] = player['understat_id'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('player_team')] = fpl_data['player_team'].values[0]



            # print(gwk, fpl_data['price'].values[0], fpl_data['position'].values[0])

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ season +'/merged_extras_xP/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        merged_player.to_csv(new_col_dir + path, index=False)

In [7]:
def add_rolling_avgs_3(season):
    print('================> starting to roll by 3 for season 20'+season)
    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_xP', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_xP/'+ path,sep=',', skipinitialspace=True)

        prev = [
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)]
            ]
        gwks = [1]

        features = ['clean_sheets', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'goals_conceded', 'goals_scored', 'ict_index',
                    'influence', 'creativity', 'threat', 'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'yellow_cards', 'saves', 'starts',
                    'team_a_score', 'team_h_score', 'total_points', 'goals', 'shots', 'xG', 'xA', 'assists_y', 'key_passes', 'npg', 'npxG', 'xGChain',  'xGBuildup',  'xP', 'selected'
                    ]

        def rolling(row):
            row_items = [row.get(col, 0) for col in features]

            if row['event'] - gwks[0] == 0:
                del prev[3]
                prev.append(row_items)

            elif row['event'] - gwks[0] == 1:
                del prev[0]
                prev.append(row_items)
                gwks[0] = row['event']

            elif row['event'] - gwks[0] == 2:
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']

            else:
                del prev[0]
                del prev[0]
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']

            # print(row['event'], len(pd.Series([round((x+y+z)/3 ,2) for x,y,z in zip(prev[0], prev[1], prev[2])])),  prev[0], prev[1], prev[2], prev[3]) #
            return pd.Series([round((x+y+z) ,2) for x,y,z in zip(prev[0], prev[1], prev[2])])
        player[[f"{col}_3" for col in features]] = player.apply(rolling, axis=1)

        # divide value by 10 to get player price
        player['value'] = player['value'] / 10
        # Save the updated DataFrame with the new columns
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled_3/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

def add_rolling_avgs_5(season):
    print('================> starting to roll by 5 for  season 20'+season)
    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_rolled_3', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_rolled_3/'+ path,sep=',', skipinitialspace=True)

        prev = [
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)],
                [0 for i in range(34)]
            ]
        gwks = [1]

        features = ['clean_sheets', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'goals_conceded', 'goals_scored', 'ict_index',
                    'influence', 'creativity', 'threat', 'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'yellow_cards', 'saves', 'starts',
                    'team_a_score', 'team_h_score', 'total_points', 'goals', 'shots', 'xG', 'xA', 'assists_y', 'key_passes', 'npg', 'npxG', 'xGChain',  'xGBuildup',  'xP', 'selected'
                    ]

        def rolling(row):
            row_items = [row.get(col, 0) for col in features]

            if row['event'] - gwks[0]== 0:
                del prev[5]
                prev.append(row_items)

            elif row['event'] - gwks[0] == 1:
                del prev[0]
                prev.append(row_items)
                gwks[0] = row['event']

            elif row['event'] - gwks[0] == 2:
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']

            elif row['event'] - gwks[0] == 3:
                del prev[0]
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']

            elif row['event'] - gwks[0] == 4:
                del prev[0]
                del prev[0]
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']
            else:
                del prev[0]
                del prev[0]
                del prev[0]
                del prev[0]
                del prev[0]
                del prev[0]
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append([0 for i in range(34)])
                prev.append(row_items)
                gwks[0] = row['event']

            return pd.Series([round((v+w+x+y+z) ,2) for v, w, x,y,z in zip(prev[0], prev[1], prev[2], prev[3], prev[4])])
        player[[f"{col}_5" for col in features]] = player.apply(rolling, axis=1)

        # Save the updated DataFrame with the new columns
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled_5/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

def add_rolling_avgs(season):
    add_rolling_avgs_3(season)
    add_rolling_avgs_5(season)

    print('<<<<================ Done rolling season 20'+season)

In [ ]:
def owenership_change(season, gwk=None):
    print('================> starting to add ownership change for season 20'+season)

    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        player_data = None
        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            players = data['elements']  # Extract the list of players
            # print(players[0]['transfers_in_event'], players[0]['transfers_out_event'])
            # transfers_in = players['transfers_in_event']
            # transfers_out = players['transfers_out_event']

            transfers = [{"id": player["id"], "transfers_in": player["transfers_in_event"], "transfers_out": player["transfers_out_event"]} for player in players]
        else:
            print('Failed to retrieve data')

        transfers_df = pd.DataFrame(transfers)

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_rolled_5', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_rolled_5/'+ path,sep=',', skipinitialspace=True)
        player['ownership_change'] = player['selected'].diff().fillna(0)

        def ownership_change(row):
            net_transfers = row['transfers_in'] - row['transfers_out']
            total_transfers = row['transfers_in'] + row['transfers_out']
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            return net_transfers_pct

        player['percenatge_net_transfers'] = player.apply(ownership_change, axis=1)

        if gwk:
            player_id = int(player.iloc[-2, player.columns.get_loc('element')])
            transfer_data = transfers_df[transfers_df['id'] == player_id]
            net_transfers = transfer_data['transfers_in'].values[0] - transfer_data['transfers_out'].values[0]
            total_transfers = transfer_data['transfers_in'].values[0] + transfer_data['transfers_out'].values[0]
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            player.iloc[-1, player.columns.get_loc('percenatge_net_transfers')] = net_transfers_pct

        # Save the updated DataFrame with the new columns
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled_5_net_transfers/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

In [ ]:
def odds(sns, nxt_gw=0):
    print('================> starting adding odds for sns 20'+sns)
    # Load the data data once to avoid redundant file reads
    data = pd.read_csv('./data/odds/E0 '+ sns +'.csv')

    data = data.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

    players_paths = next(os.walk('./data/joint/'+ sns +'/merged_extras_rolled_5_net_transfers', [None], [None],[]))[2]
    for path in players_paths:
        rolled = pd.read_csv('./data/joint/'+ sns +'/merged_extras_rolled_5_net_transfers/'+ path,sep=',', skipinitialspace=True)

        def add_odds(row, data):
            # Filter the data DataFrame for the matching teams
            # print(data)
            match = data[(data['h_team'] == row['h_team']) & (data['a_team'] == row['a_team'])]
            # Check if a match is found

            if not match.empty:
                # print(row['value'])

                # Extract the relevant data values
                # Convert the data to probabilities
                odds_ = match.iloc[0]

                WHH = round(1/odds_['WHH'], 5)
                WHD = round(1/odds_['WHD'], 5)
                WHA = round(1/odds_['WHA'], 5)

                # Normalize the probabilities (to make the probabilities sum to 100%)
                WH_sum = WHH + WHD + WHA
                WHH_ = round(WHH/WH_sum, 3)
                WHD_ = round(WHD/WH_sum, 3)
                WHA_ = round(WHA/WH_sum, 3)

                pts_bps = row['total_points'] - row['bonus']
                return pd.Series([pts_bps, WHH_, WHD_, WHA_])
            else:
                # Return NaN for rows with no match
                return pd.Series([None,None, None, None])

        # Apply the function to the 'rolled' DataFrame
        rolled[['pts_bps','whh', 'whd', 'wha']] = rolled.apply(add_odds, axis=1, data=data)

        if nxt_gw:
            odds_nxt = pd.read_csv(f'./data/odds/odds_{nxt_gw}.csv')
            opp_team = rolled['opponent_team']
            player_team = rolled['player_team'].loc[0]
            # print(row)

            team_odds_nxt = odds_nxt[(odds_nxt['h_team'] == player_team) | (odds_nxt['a_team']==player_team)]

            if(team_odds_nxt.empty):
                continue


            h_team = team_odds_nxt.loc[:, 'h_team'].values[0]
            a_team = team_odds_nxt.loc[:, 'a_team'].values[0]
            whh = team_odds_nxt.loc[:,'WHH'].values[0]
            whd = team_odds_nxt.loc[:,'WHD'].values[0]
            wha = team_odds_nxt.loc[:,'WHA'].values[0]
            h_fdr = team_odds_nxt.loc[:,'h_fdr'].values[0]
            a_fdr = team_odds_nxt.loc[:,'a_fdr'].values[0]
            was_home = True if h_team == player_team else False

            rolled.iloc[-1, rolled.columns.get_loc('whh')] = whh
            rolled.iloc[-1, rolled.columns.get_loc('whd')] = whd
            rolled.iloc[-1, rolled.columns.get_loc('wha')] = wha
            rolled.iloc[-1, rolled.columns.get_loc('was_home')] = was_home
            rolled.iloc[-1, rolled.columns.get_loc('h_team')] = h_team
            rolled.iloc[-1, rolled.columns.get_loc('a_team')] = a_team
            rolled.iloc[-1, rolled.columns.get_loc('team_h_difficulty')] = h_fdr
            rolled.iloc[-1, rolled.columns.get_loc('team_a_difficulty')] = a_fdr

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ sns +'/merged_extras_odds/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        rolled.to_csv(new_col_dir + path, index=False)

In [ ]:
def merge_files(season):
    print('starting to merge files for 20'+ season)
    paths = next(os.walk('./data/joint/'+ season +'/merged_extras_odds', [None], [None],[]))[2]
    print(len(paths))
    files_list = [pd.read_csv('./data/joint/'+ season +'/merged_extras_odds/' + path)  for  path in paths ]
    merged_files = pd.concat(files_list)

    # print(merged_files['fpl_id'])
    # Save the new DataFrame
    new_col_dir = './data/joint/'+ season +'/'

    merged_files.to_csv(new_col_dir  +'merged_player_data.csv', index=False)

In [16]:
# For the current season
%run "./getplayerdetails_24_25.ipynb"